# Bairro em Dia — Treinamento de Machine Learning

**Pergunta:** com dados disponíveis no momento do registro, estimar o risco de uma solicitação urbana ultrapassar o prazo de referência para resolução.

> Execute em ordem. Primeiro inspecione os nomes reais das colunas. Não preencha métricas no relatório antes de executar este notebook com dados públicos reais. A versão do dataset, período, recurso e data de download devem ser registrados em `../dados/README.md`.

Modelo: Random Forest, com pré-processamento dentro de `Pipeline`, validação cruzada somente no treino e avaliação final no teste.

In [ ]:
from pathlib import Path
import warnings, json, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, learning_curve
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, classification_report, ConfusionMatrixDisplay,
                             RocCurveDisplay)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
DATA_PATH = Path("../dados/solicitacoes_sp156.csv")
PRAZO_REFERENCIA_DIAS = 30  # AJUSTAR após analisar a distribuição e justificar no relatório

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {DATA_PATH.resolve()}. Baixe o CSV público do SP156 "
        "e salve como dados/solicitacoes_sp156.csv. Não substitua por dados sintéticos."
    )

df_raw = pd.read_csv(DATA_PATH, low_memory=False)
print("Dimensão:", df_raw.shape)
display(df_raw.head())
print("Colunas disponíveis:")
print(df_raw.columns.tolist())

## 1. Mapeamento das colunas reais

Preencha os valores abaixo com os nomes exatos mostrados na célula anterior e confirmados no dicionário de dados. Se uma coluna não existir, não invente: avalie outro recorte/recurso ou reformule a pergunta de ML.

In [ ]:
# TODO OBRIGATÓRIO: substituir None pelos nomes reais do CSV.
COLUNAS = {
    "data_abertura": None,   # ex.: data de abertura da solicitação
    "data_fechamento": None, # data de encerramento/fechamento efetivo
    "categoria": None,       # assunto/serviço/categoria
    "regiao": None,          # distrito/prefeitura regional/subprefeitura
    "canal": None            # canal de entrada, se disponível
}

faltantes = [k for k, v in COLUNAS.items() if v is None]
if faltantes:
    raise ValueError("Mapeie as colunas reais antes de continuar: " + ", ".join(faltantes))
inexistentes = [v for v in COLUNAS.values() if v not in df_raw.columns]
if inexistentes:
    raise ValueError(f"Colunas mapeadas não encontradas no CSV: {inexistentes}")

df = df_raw.copy()
df["data_abertura"] = pd.to_datetime(df[COLUNAS["data_abertura"]], errors="coerce", dayfirst=True)
df["data_fechamento"] = pd.to_datetime(df[COLUNAS["data_fechamento"]], errors="coerce", dayfirst=True)
df["categoria"] = df[COLUNAS["categoria"]].astype("string").str.strip()
df["regiao"] = df[COLUNAS["regiao"]].astype("string").str.strip()
if COLUNAS["canal"]:
    df["canal"] = df[COLUNAS["canal"]].astype("string").str.strip()
else:
    df["canal"] = "NAO_INFORMADO"

# Usar apenas solicitações com abertura e fechamento válidos.
df = df.dropna(subset=["data_abertura", "data_fechamento", "categoria", "regiao"]).copy()
df = df.drop_duplicates()
df["tempo_resolucao_dias"] = (df["data_fechamento"] - df["data_abertura"]).dt.total_seconds() / 86400
df = df[df["tempo_resolucao_dias"] >= 0].copy()
if len(df) < 100:
    raise ValueError(f"Apenas {len(df)} registros válidos. Verifique o recorte/dados; são necessários dados suficientes para avaliação.")

print("Registros após limpeza:", len(df))
display(df["tempo_resolucao_dias"].describe())
df["tempo_resolucao_dias"].hist(bins=40)
plt.title("Distribuição do tempo de resolução")
plt.xlabel("Dias")
plt.ylabel("Quantidade")
plt.show()

## 2. Criação do alvo e prevenção de vazamento

O prazo de referência precisa ser justificado no relatório. O alvo usa o tempo entre abertura e fechamento, mas os preditores abaixo usam apenas categoria, região, mês/dia da abertura e canal. **A data de fechamento e o tempo de resolução não entram em X.**

In [ ]:
df["atrasou"] = (df["tempo_resolucao_dias"] > PRAZO_REFERENCIA_DIAS).astype(int)
df["mes"] = df["data_abertura"].dt.month.astype(str)
df["dia_semana"] = df["data_abertura"].dt.dayofweek.add(1).astype(str)

FEATURES = ["categoria", "regiao", "mes", "dia_semana", "canal"]
X = df[FEATURES].copy()
y = df["atrasou"].copy()

print("Distribuição do alvo:")
display(y.value_counts().rename(index={0:"Dentro do prazo de referência",1:"Atrasou"}))
display(y.value_counts(normalize=True).rename("proporção"))

if y.nunique() != 2 or y.value_counts().min() < 5:
    raise ValueError("O alvo não possui exemplos suficientes em ambas as classes. Revise prazo/recorte.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Treino:", X_train.shape, "Teste:", X_test.shape)

## 3. Pipeline e busca de hiperparâmetros

O OneHotEncoder é ajustado dentro do pipeline em cada fold, evitando que informações do fold de validação vazem para o treinamento. A grade abaixo contém 36 combinações de três hiperparâmetros. A validação cruzada ocorre somente em `X_train`.

In [ ]:
categorical_features = FEATURES
preprocessor = ColumnTransformer(
    transformers=[("categorical", Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)],
    remainder="drop"
)

pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", RandomForestClassifier(random_state=RANDOM_STATE, class_weight="balanced"))
])

param_grid = {
    "clf__n_estimators": [100, 200, 300],
    "clf__max_depth": [5, 10, None],
    "clf__min_samples_leaf": [1, 5, 10]
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
grid = GridSearchCV(pipeline, param_grid, scoring="f1", cv=cv, n_jobs=-1, return_train_score=True)
grid.fit(X_train, y_train)

print("Combinações testadas:", len(grid.cv_results_["params"]))
print("Melhores parâmetros:", grid.best_params_)
print("Melhor F1 médio CV:", round(grid.best_score_, 4))
resultados_grid = pd.DataFrame(grid.cv_results_).sort_values("rank_test_score")
display(resultados_grid[["params","mean_train_score","mean_test_score","std_test_score","rank_test_score"]].head(15))

## 4. Avaliação treino x teste e baseline

O teste é utilizado somente agora, depois da seleção de hiperparâmetros. Accuracy não deve ser interpretada isoladamente se as classes estiverem desbalanceadas.

In [ ]:
best_model = grid.best_estimator_

def metricas(model, X_part, y_part, nome):
    pred = model.predict(X_part)
    prob = model.predict_proba(X_part)[:, 1]
    return {
        "Conjunto": nome,
        "Accuracy": accuracy_score(y_part, pred),
        "Precision": precision_score(y_part, pred, zero_division=0),
        "Recall": recall_score(y_part, pred, zero_division=0),
        "F1": f1_score(y_part, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_part, prob) if y_part.nunique() == 2 else np.nan
    }

metricas_modelo = pd.DataFrame([
    metricas(best_model, X_train, y_train, "Treino"),
    metricas(best_model, X_test, y_test, "Teste")
])
display(metricas_modelo.round(4))

baseline = DummyClassifier(strategy="prior")
baseline.fit(X_train, y_train)
print("Baseline — teste:")
print(classification_report(y_test, baseline.predict(X_test), zero_division=0))
print("Modelo — teste:")
print(classification_report(y_test, best_model.predict(X_test), zero_division=0))

In [ ]:
fig, ax = plt.subplots(figsize=(6,5))
ConfusionMatrixDisplay.from_estimator(best_model, X_test, y_test, ax=ax, cmap="Blues")
ax.set_title("Matriz de confusão — teste")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6,5))
RocCurveDisplay.from_estimator(best_model, X_test, y_test, ax=ax)
ax.set_title("Curva ROC — teste")
plt.tight_layout()
plt.show()

## 5. Overfitting e importância das variáveis

Compare o F1 de treino com o F1 de teste e o resultado médio da validação cruzada. Uma diferença alta pode indicar overfitting. A importância por permutação é calculada no teste e deve ser interpretada com cautela.

In [ ]:
scores = pd.DataFrame({
    "Conjunto": ["Treino", "CV (treino)", "Teste"],
    "F1": [
        f1_score(y_train, best_model.predict(X_train), zero_division=0),
        grid.best_score_,
        f1_score(y_test, best_model.predict(X_test), zero_division=0)
    ]
})
display(scores)
scores.plot(x="Conjunto", y="F1", kind="bar", legend=False, ylim=(0,1), figsize=(7,4), title="F1: treino, validação cruzada e teste")
plt.ylabel("F1")
plt.tight_layout()
plt.show()

perm = permutation_importance(best_model, X_test, y_test, n_repeats=8, random_state=RANDOM_STATE, scoring="f1")
importance = pd.Series(perm.importances_mean, index=FEATURES).sort_values()
display(importance.to_frame("importancia_media_permutacao"))
importance.plot(kind="barh", figsize=(7,4), title="Importância por permutação — teste")
plt.xlabel("Queda média de F1 ao embaralhar")
plt.tight_layout()
plt.show()

## 6. Salvar artefato e registrar métricas

O artefato é o pipeline completo (pré-processamento + classificador). O serviço FastAPI recebe `categoria`, `regiao`, `mes`, `dia_semana` e `canal`, que correspondem às features usadas no treinamento.

In [ ]:
MODEL_DIR = Path("model")
MODEL_DIR.mkdir(exist_ok=True)
MODEL_FILE = MODEL_DIR / "random_forest.joblib"
joblib.dump(best_model, MODEL_FILE)
print("Modelo salvo em:", MODEL_FILE.resolve())

metrics_file = MODEL_DIR / "metricas_teste.json"
metrics_file.write_text(json.dumps(metricas_modelo.to_dict(orient="records"), indent=2), encoding="utf-8")
print("Métricas salvas em:", metrics_file.resolve())

# Exemplo de inferência local:
exemplo = pd.DataFrame([{
    "categoria": str(X_train.iloc[0]["categoria"]),
    "regiao": str(X_train.iloc[0]["regiao"]),
    "mes": str(X_train.iloc[0]["mes"]),
    "dia_semana": str(X_train.iloc[0]["dia_semana"]),
    "canal": str(X_train.iloc[0]["canal"])
}])
print("Probabilidade estimada de atraso:", round(float(best_model.predict_proba(exemplo)[0,1]), 4))